In [1]:
!rm -rf diploma_centpy_parallelization_py
# Флаг -b указывает конкретную ветку
!git clone -b feature/jax-centpy https://github.com/filkinc/diploma_centpy_parallelization_py.git
%cd diploma_centpy_parallelization_py
%cd /content/diploma_centpy_parallelization_py/jax_centpy

# Установка зависимостей
!pip install centpy pandas matplotlib seaborn

Cloning into 'diploma_centpy_parallelization_py'...
remote: Enumerating objects: 185, done.
remote: Counting objects: 100% (185/185), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 185 (delta 72), reused 157 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (185/185), 19.99 MiB | 17.01 MiB/s, done.
Resolving deltas: 100% (72/72), done.
/content/diploma_centpy_parallelization_py
/content/diploma_centpy_parallelization_py/jax_centpy


In [2]:
import os
import time
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import numpy as np

from core import Pars2d, Equation2d
from solver import Solver2d, FastSolver2d
from boundaries import periodic_bc_2d
from equations import make_euler_riemann_2d
from schemes import compute_rhs_sd2_2d
from limiters import monotonized_central, minmod

In [3]:
def run_gpu_benchmark():
    print(f"JAX Device(s): {jax.devices()}")
    J = 200

    pars = Pars2d(
        x_init=0.0, x_final=1.0, y_init=0.0, y_final=1.0,
        t_final=0.4, dt_out=0.005, Jx=J, Jy=J, cfl=0.475, scheme="sd2"
    )

    eqn = make_euler_riemann_2d()
    solver = FastSolver2d(pars, eqn, limiter_name="minmod")
    solverForPlot = Solver2d(pars, eqn, scheme_name="sd2", limiter_name="minmod")

    print("--- Прогрев JAX (Компиляция XLA) ---")
    # Прогреваем ТОЛЬКО первый шаг решателя (именно он содержит всю тяжелую математику)
    x_1d = jnp.linspace(pars.x_init + pars.dx / 2, pars.x_final - pars.dx / 2, pars.Jx)
    y_1d = jnp.linspace(pars.y_init + pars.dy / 2, pars.y_final - pars.dy / 2, pars.Jy)
    X, Y = jnp.meshgrid(x_1d, y_1d, indexing='ij')
    test_u = eqn.initial_data(X, Y)
    _ = solver.update_step_jit(0.0, test_u, 0.001).block_until_ready()
    print("Прогрев завершен.\n")

    print(f"--- Запуск JAX GPU Бенчмарка (Euler 2D Riemann, {J}x{J}) ---")
    t0 = time.time()
    results = solver.solve()
    results['u'][-1].block_until_ready()
    t1 = time.time()

    print(f"\n[GPU JAX] Чистое время выполнения: {t1 - t0:.4f} секунд")

    resultsForPlot = solverForPlot.solve()

    return resultsForPlot, pars

if __name__ == "__main__":
    jax.config.update("jax_enable_x64", True)
    soln, pars = run_gpu_benchmark()

JAX Device(s): [CudaDevice(id=0)]
--- Прогрев JAX (Компиляция XLA) ---
Прогрев завершен.

--- Запуск JAX GPU Бенчмарка (Euler 2D Riemann, 200x200) ---
Starting 2D simulation: Euler 2D (Riemann)
Grid: 200x200, Scheme: SD2/minmod

[GPU JAX] Чистое время выполнения: 1.7479 секунд
Starting 2D simulation: Euler 2D (Riemann)
Grid: 200x200, Scheme: SD2/minmod


In [4]:
u_data = soln['u']
X_grid = soln['X']
Y_grid = soln['Y']

# Инициализация фигуры и осей, берем границы из объекта pars
fig, ax = plt.subplots()
ax.set_xlim(pars.x_init, pars.x_final)
ax.set_ylim(pars.y_init, pars.y_final)

# Для первого кадра (плотность, так как индекс [..., 0] соответствует плотности в уравнениях Эйлера)
data_init = u_data[0, ..., 0]

# 1. Создаем фоновую тепловую карту с помощью imshow
im = ax.imshow(
    data_init.T, # Транспонируем, так как imshow ожидает порядок (y, x), а у вас индексация 'ij'
    extent=[pars.x_init, pars.x_final, pars.y_init, pars.y_final],
    origin='lower',
    cmap='coolwarm',            # Золотой стандарт для волновых процессов
    interpolation='bicubic',
    aspect='auto'
)

cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Плотность (u[..., 0])')

# 2. Отрисовываем начальные контуры поверх тепловой карты
ax.contour(
    X_grid, Y_grid, data_init,
    levels=20,
    colors='black',
    alpha=0.5,
    linewidths=0.5
)

# Функция обновления для каждого кадра анимации
def animate(i):
    # Получаем данные текущего шага
    data = u_data[i, ..., 0]

    # Обновляем данные на тепловой карте
    im.set_data(data.T)

    # Динамически обновляем границы цветовой шкалы (опционально)
    im.set_clim(vmin=data.min(), vmax=data.max())

    # Удаляем старые линии контуров из коллекции осей
    for c in ax.collections:
        c.remove()

    # Рисуем новые контурные линии
    ax.contour(
        X_grid, Y_grid, data,
        levels=20,
        colors='black',
        alpha=0.5,
        linewidths=0.5
    )

    return [im]

plt.close() # Закрываем статичную фигуру

# Создаем анимацию (количество кадров равно размеру массива времени)
num_frames = u_data.shape[0]
anim = animation.FuncAnimation(fig, animate, frames=num_frames, interval=100, blit=False)

# Выводим как HTML5 видео
HTML(anim.to_html5_video())